# Preprocess Dosage Sensitivity Region and Gene Data
* Dosage sensitivity data from [ClinGen](https://search.clinicalgenome.org/kb/downloads)
* Used the Archived 2025-07-29 dosage sensitivity [gene](https://ftp.clinicalgenome.org/archive/20250729/ClinGen_gene_curation_list_GRCh38.tsv) and [region](https://ftp.clinicalgenome.org/archive/20250729/ClinGen_region_curation_list_GRCh38.tsv) files with haploinsufficiency (HI) and triplosensitivity (TS) scores equal to 3 ("sufficient evidence" tier)

In [1]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import plotly.graph_objects as go
from pathlib import Path
from urllib.request import urlretrieve

In [2]:
supporting_data_dir = Path("supporting_data")
supporting_data_dir.mkdir(exist_ok=True)

dosage_gene_url = "https://ftp.clinicalgenome.org/archive/20250729/ClinGen_gene_curation_list_GRCh38.tsv"
dosage_region_url = "https://ftp.clinicalgenome.org/archive/20250729/ClinGen_region_curation_list_GRCh38.tsv"

dosage_gene_path = supporting_data_dir / "ClinGen_gene_curation_list_GRCh38.tsv"
dosage_region_path = supporting_data_dir / "ClinGen_region_curation_list_GRCh38.tsv"

files = {
    dosage_gene_url: dosage_gene_path,
    dosage_region_url: dosage_region_path
}

for url, output_path in files.items():
    if output_path.exists():
        print(f"Already exists: {output_path}")
    else:
        print(f"Downloading {output_path.name}...")
        urlretrieve(url, output_path)

print("Done.")

Already exists: supporting_data/ClinGen_gene_curation_list_GRCh38.tsv
Already exists: supporting_data/ClinGen_region_curation_list_GRCh38.tsv
Done.


In [3]:
dosage_gene_df = (
    pd.read_csv(dosage_gene_path, delimiter='\t', header=5)
    .rename(columns=lambda col: col.strip('#')) #remove comment character from header
)
dosage_region_df = (
    pd.read_csv(dosage_region_path, delimiter='\t', header=5)
    .rename(columns=lambda col: col.strip('#'))
)

In [4]:
dosage_gene_df['feature_type'] = 'gene'
dosage_region_df['feature_type'] = 'region'

dosage_gene_df['feature_name'] = dosage_gene_df['Gene Symbol']
dosage_gene_df['feature_id'] = dosage_gene_df['Gene ID']

dosage_region_df['feature_name'] = dosage_region_df['ISCA Region Name']
dosage_region_df['feature_id'] = dosage_region_df['ISCA ID']

dosage_df = pd.concat(
    [dosage_gene_df, dosage_region_df]
)
len(dosage_df)

2114

In [5]:
dosage_df = dosage_df.query("`Genomic Location` != 'tbd'")
len(dosage_df)

2106

In [6]:
dosage_df['chromosome'] = dosage_df['Genomic Location'].str.extract(
    r"chr([\dXY]+)\:"
)

dosage_df['genomic_start'] = dosage_df['Genomic Location'].str.extract(
    r"\:(\d+)\-"
).astype(int)
dosage_df['genomic_stop'] = dosage_df['Genomic Location'].str.extract(
    r"\d+\-(\d+)"
).astype(int)

In [7]:
dosage_df['Haploinsufficiency Score'] = (
    dosage_df['Haploinsufficiency Score']
    .fillna(-1)
    .astype(int)
)

dosage_df['Haploinsufficiency Description'] = (
    dosage_df['Haploinsufficiency Description']
    .fillna('None')
)

dosage_df['Triplosensitivity Score'] = (
    dosage_df['Triplosensitivity Score']
    .fillna(-1)
    .replace("Not yet evaluated", -1)
    .astype(int)
)

dosage_df['Triplosensitivity Description'] = (
    dosage_df['Triplosensitivity Description']
    .fillna('None')
)

In [8]:
(
    dosage_df
    .groupby(['feature_type', 'Haploinsufficiency Score', 'Haploinsufficiency Description'])
    .size()
    .unstack(level=0, fill_value=0)
)

,feature_type,gene,region
Haploinsufficiency Score,Haploinsufficiency Description,,
-1,None,0,2
0,No evidence available,289,91
1,Little evidence for dosage pathogenicity,134,8
2,Some evidence for dosage pathogenicity,10,6
3,Sufficient evidence for dosage pathogenicity,397,48
30,Gene associated with autosomal recessive phenotype,731,3
40,Dosage sensitivity unlikely,35,352


In [9]:
(
    dosage_df
    .groupby(['feature_type', 'Triplosensitivity Score', 'Triplosensitivity Description'])
    .size()
    .unstack(level=0, fill_value=0)
)

feature_type                                                          gene  \
Triplosensitivity Score Triplosensitivity Description                        
-1                      None                                             1   
                        Not yet evaluated                              272   
 0                      No evidence available                         1308   
 1                      Little evidence for dosage pathogenicity         9   
 2                      Some evidence for dosage pathogenicity           1   
 3                      Sufficient evidence for dosage pathogenicity     2   
 40                     Dosage sensitivity unlikely                      3   

feature_type                                                          region  
Triplosensitivity Score Triplosensitivity Description                         
-1                      None                                               1  
                        Not yet evaluated                                  0  
 0                      No evidence available                            368  
 1                      Little evidence for dosage pathogenicity          19  
 2                      Some evidence for dosage pathogenicity            10  
 3                      Sufficient evidence for dosage pathogenicity      20  
 40                     Dosage sensitivity unlikely                       92

In [10]:
feat_df_hi = (
    dosage_df
    .query("`Haploinsufficiency Score` == 3")
)[['feature_type', 'feature_name', 'feature_id', 'chromosome', 'genomic_start', 'genomic_stop']]
len(feat_df_hi)

445

In [11]:
feat_df_ts = (
    dosage_df
    .query("`Triplosensitivity Score` == 3") # score is a string type in this table
)[['feature_type', 'feature_name', 'feature_id', 'chromosome', 'genomic_start', 'genomic_stop']]
len(feat_df_ts)

22

In [12]:
feat_df_hi.to_csv(
    "supporting_data/haploinsufficiency-regions-preprocessed.csv",
    index=False
)

feat_df_ts.to_csv(
    "supporting_data/triplosensitivity-regions-preprocessed.csv",
    index=False
)